# Part 2 - Original-Session Exploration

This notebook treats the existing `session_id` as the original logging-session identifier. Any
derived clean-session boundary is stored separately and never overwrites `session_id`.


## Configuration and imports


In [3]:
from pathlib import Path
import json
import sys
import numpy as np
import pandas as pd

WORKING_DIR = Path.cwd()
REPO_ROOT = WORKING_DIR if (WORKING_DIR / "data" / "final_clean_events.csv").exists() else WORKING_DIR.parent
sys.path.insert(0, str(REPO_ROOT))

from utils.exact_event_analysis import (
    ACTION_EVENT_TYPES,
    INACTIVITY_THRESHOLDS_MINUTES,
    MISSING_SENTINEL,
    REQUIRED_COLUMNS,
    SELECTED_INACTIVITY_THRESHOLD_MINUTES,
    VIEW_EVENT_TYPES,
    add_duration_fields,
    add_exact_tokens,
    add_time_fields,
    add_token_ids,
    availability_by_event_type,
    build_session_sequences,
    build_token_dictionary,
    component_by_event_type,
    construct_clean_sessions,
    exact_tuple_counts,
    file_fingerprint,
    inferred_dtypes,
    input_csv_path,
    markdown_table,
    masked_sample,
    missing_summary,
    original_order_timestamp_issues,
    original_session_summary,
    quantile_table,
    read_events,
    representative_sequences,
    safe_json,
    save_token_outputs,
    sort_events,
    threshold_comparison,
    validate_required_columns,
    validate_tokenization,
    value_counts_with_pct,
)

pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 160)
np.random.seed(42)

INPUT_CSV = input_csv_path(REPO_ROOT)
print(f"Repository root: {REPO_ROOT}")
print(f"Input CSV: {INPUT_CSV.relative_to(REPO_ROOT)}")


Repository root: D:\BehaviourClassification
Input CSV: data\final_clean_events.csv


## Load, validate, parse time, and deterministically sort events


In [5]:
before_fingerprint = file_fingerprint(INPUT_CSV)
events = read_events(INPUT_CSV)
print(f"Rows after load: {len(events):,}")
validate_required_columns(events)
events, time_metadata = add_time_fields(events)
events = add_duration_fields(events)
print(f"Rows after derived time/duration fields: {len(events):,}")
sorted_events = sort_events(events)
sorted_events = construct_clean_sessions(sorted_events, SELECTED_INACTIVITY_THRESHOLD_MINUTES)
print(f"Rows after deterministic sorting and clean-session derivation: {len(sorted_events):,}")
print(safe_json({
    "selected_inactivity_threshold_minutes": SELECTED_INACTIVITY_THRESHOLD_MINUTES,
    "thresholds_compared_minutes": INACTIVITY_THRESHOLDS_MINUTES,
    "primary_event_time_column": time_metadata["primary_event_time_column"],
}))


Rows after load: 114,534
Rows after derived time/duration fields: 114,534
Rows after deterministic sorting and clean-session derivation: 114,534
{
  "selected_inactivity_threshold_minutes": 30,
  "thresholds_compared_minutes": [
    15,
    30,
    60
  ],
  "primary_event_time_column": "timestamp_datetime"
}


## Original logging-session statistics

Session spans and inter-event gaps are calculated after deterministic ordering by `session_id`,
primary event time, secondary event time, `record_id`, and source row number.


In [7]:
sessions = original_session_summary(sorted_events)
print(f"Original sessions: {len(sessions):,}")
print("\nEvent-count distribution:")
print(markdown_table(quantile_table(sessions["event_count"], [0, .25, .5, .75, .9, .95, .99, 1]), max_rows=20))
print("\nSession-span distribution in seconds:")
print(markdown_table(quantile_table(sessions["session_span_seconds"], [0, .25, .5, .75, .9, .95, .99, 1]), max_rows=20))
print("\nInter-event gap distribution in seconds:")
print(markdown_table(quantile_table(sorted_events["time_gap_from_previous_seconds"], [0, .25, .5, .75, .9, .95, .99, 1]), max_rows=20))
print("\nSessions with exceptionally large event counts:")
print(markdown_table(sessions.sort_values("event_count", ascending=False).head(10), max_rows=10))
print("\nSessions with exceptionally long spans:")
print(markdown_table(sessions.sort_values("session_span_seconds", ascending=False).head(10), max_rows=10))


Original sessions: 1,425

Event-count distribution:
| quantile | value |
| --- | --- |
| 0.0 | 1.0 |
| 0.25 | 15.0 |
| 0.5 | 34.0 |
| 0.75 | 87.0 |
| 0.9 | 191.60000000000014 |
| 0.95 | 301.5999999999999 |
| 0.99 | 725.76 |
| 1.0 | 1524.0 |

Session-span distribution in seconds:
| quantile | value |
| --- | --- |
| 0.0 | 0.0 |
| 0.25 | 36.663 |
| 0.5 | 257.368 |
| 0.75 | 1732.088 |
| 0.9 | 7563.691000000006 |
| 0.95 | 20235.319599999995 |
| 0.99 | 93387.03088 |
| 1.0 | 321112.548 |

Inter-event gap distribution in seconds:
| quantile | value |
| --- | --- |
| 0.0 | 0.0 |
| 0.25 | 0.003 |
| 0.5 | 0.037 |
| 0.75 | 1.439 |
| 0.9 | 7.135 |
| 0.95 | 26.131199999999982 |
| 0.99 | 337.5910399999999 |
| 1.0 | 320928.179 |

Sessions with exceptionally large event counts:
| original_session_id | event_count | session_start | session_end | non_null_device_count | non_null_customer_count | max_inter_event_gap_seconds | session_span_seconds |
| --- | --- | --- | --- | --- | --- | --- | --- |
| D01C

## Identifier and timestamp integrity checks

Known-to-known device/customer changes are examined. Changes between a known value and a missing
value are not treated as split conditions.


In [9]:
issues = original_order_timestamp_issues(events)
multi_device = sessions.loc[sessions["non_null_device_count"].gt(1)].sort_values("non_null_device_count", ascending=False)
multi_customer = sessions.loc[sessions["non_null_customer_count"].gt(1)].sort_values("non_null_customer_count", ascending=False)
gap_counts = {
    f"gaps_exceeding_{threshold}_minutes": int(sorted_events["time_gap_from_previous_seconds"].gt(threshold * 60).sum())
    for threshold in [15, 30, 60]
}
print("Timestamp issues before sorting:")
printable_issues = {k: v for k, v in issues.items() if not isinstance(v, pd.DataFrame)}
print(safe_json(printable_issues))
print("\nSample decreasing rows before sorting:")
print(markdown_table(issues["sample_decreasing_rows"], max_rows=10))
print("\nSessions containing more than one non-null device:")
print(markdown_table(multi_device.head(10), max_rows=10))
print("\nSessions containing more than one non-null customer:")
print(markdown_table(multi_customer.head(10), max_rows=10))
print("\nInter-event gap counts:")
print(safe_json(gap_counts))


Timestamp issues before sorting:
{
  "invalid_primary_event_time_rows": 0,
  "invalid_or_missing_pair_rows": 0,
  "decreasing_timestamp_rows_before_sort": 6101
}

Sample decreasing rows before sorting:
| record_id | session_id | primary_event_time | source_row_number |
| --- | --- | --- | --- |
| 6a471aa6a831463d3a03962f | 000A5106-BBC0-45BA-AC04-53BAF35B99F3 | 2026-07-03 02:12:52.419000+00:00 | 62018 |
| 6a471ae12eed2dfdb6c9c2ed | 000A5106-BBC0-45BA-AC04-53BAF35B99F3 | 2026-07-03 02:13:51.685000+00:00 | 62055 |
| 6a44d8cb0a23f20918554df3 | 0096BDF5-AFCF-4367-B5AB-6FA86F6F3CD3 | 2026-07-01 09:07:19.035000+00:00 | 20775 |
| 6a44d8cb2eed2dfdb6c975fc | 0096BDF5-AFCF-4367-B5AB-6FA86F6F3CD3 | 2026-07-01 09:07:19.008000+00:00 | 20780 |
| 6a4b78b5598dfb494836952d | 00c93be9-5085-49d0-b84e-1a1ce50ec112 | 2026-07-06 09:43:15.566000+00:00 | 107238 |
| 6a4b78d001f1b00cb2cd52bd | 00c93be9-5085-49d0-b84e-1a1ce50ec112 | 2026-07-06 09:43:40.627000+00:00 | 107271 |
| 6a4b78fc5f185618f1c728ed | 00c93be

## Inactivity-threshold comparison and clean-session construction

The 15-, 30-, and 60-minute thresholds are compared as modeling thresholds. The selected
30-minute threshold is configurable and should not be read as a confirmed business rule.


In [11]:
threshold_table = threshold_comparison(sorted_events, INACTIVITY_THRESHOLDS_MINUTES)
clean_session_count = sorted_events["clean_session_id"].nunique(dropna=True)
split_reasons = sorted_events.loc[sorted_events["session_split_flag"], "session_split_reason"].value_counts().reset_index()
split_reasons.columns = ["session_split_reason", "count"]
print("\nThreshold comparison:")
print(markdown_table(threshold_table, max_rows=10))
print("\nClean-session summary:")
print(safe_json({
    "original_sessions": int(sessions.shape[0]),
    "clean_sessions": int(clean_session_count),
    "selected_threshold_minutes": SELECTED_INACTIVITY_THRESHOLD_MINUTES,
    "split_rows": int(sorted_events["session_split_flag"].sum()),
}))
print("\nSplit reasons:")
print(markdown_table(split_reasons, max_rows=20))
print("\nClean-session event-count distribution:")
clean_session_events = sorted_events.groupby("clean_session_id", dropna=False).size()
print(markdown_table(quantile_table(clean_session_events, [0, .25, .5, .75, .9, .95, .99, 1]), max_rows=20))



Threshold comparison:
| threshold_minutes | gaps_exceeding_threshold | percentage_of_observed_gaps | resulting_clean_sessions_if_gap_only |
| --- | --- | --- | --- |
| 15 | 544 | 0.481 | 1969 |
| 30 | 313 | 0.2767 | 1738 |
| 60 | 173 | 0.1529 | 1598 |

Clean-session summary:
{
  "original_sessions": 1425,
  "clean_sessions": 1739,
  "selected_threshold_minutes": 30,
  "split_rows": 314
}

Split reasons:
| session_split_reason | count |
| --- | --- |
| inactivity>30m | 313 |
| known_customer_changed | 1 |

Clean-session event-count distribution:
| quantile | value |
| --- | --- |
| 0.0 | 1.0 |
| 0.25 | 11.0 |
| 0.5 | 28.0 |
| 0.75 | 71.5 |
| 0.9 | 164.20000000000005 |
| 0.95 | 259.0999999999999 |
| 0.99 | 527.0599999999986 |
| 1.0 | 1518.0 |


## Part 2 summary


In [13]:
after_fingerprint = file_fingerprint(INPUT_CSV)
assert before_fingerprint == after_fingerprint, "Input CSV changed during session exploration."
print("Input file unchanged: passed")
print(safe_json({
    "original_sessions": int(sessions.shape[0]),
    "clean_sessions": int(sorted_events["clean_session_id"].nunique(dropna=True)),
    "selected_inactivity_threshold_minutes": SELECTED_INACTIVITY_THRESHOLD_MINUTES,
    "gap_threshold_counts": gap_counts,
    "timestamp_validation_status": "completed",
}))


Input file unchanged: passed
{
  "original_sessions": 1425,
  "clean_sessions": 1739,
  "selected_inactivity_threshold_minutes": 30,
  "gap_threshold_counts": {
    "gaps_exceeding_15_minutes": 544,
    "gaps_exceeding_30_minutes": 313,
    "gaps_exceeding_60_minutes": 173
  },
  "timestamp_validation_status": "completed"
}
